<a href="https://colab.research.google.com/github/AjanyaVinayan/ML-LAB-2547205/blob/lab---8/Lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 8: Implementation and Performance Evaluation of Categorical Naive Bayes Classifier

In [ ]:
# Install necessary libraries
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import numpy as np

### Data Preprocessing: Load the dataset and separate features/target.

In [ ]:
# Load the dataset from Google Sheets
# The provided link is for a public Google Sheet, so we can directly read it using pandas.
# Make sure the sheet is published to the web or accessible to 'anyone with the link'.
url = 'https://docs.google.com/spreadsheets/d/1mnoUgK8YxInzoDQ7P7iBM1HeZ8tcnUSvMU_H1OWndFA/export?format=csv&gid=0'
df = pd.read_csv(url)

display(df.head())

,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes


In [ ]:
# Separate input features (X) and target variable (y)
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']]
y = df['Play Tennis']

print("Features (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

Features (X) head:


,Outlook,Temperature,Humidity,Wind
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rain,Mild,High,Weak
4,Rain,Cool,Normal,Weak



Target (y) head:


,Play Tennis
0,No
1,No
2,Yes
3,Yes
4,Yes


### Convert all categorical feature values and target labels into numerical representations.

In [ ]:
# Fix: NameError - ensure 'X' and 'y' are defined from previous cells before running this cell.
# Initialize OrdinalEncoder for features
# We will fit the encoder on all categorical features.
# handle_unknown='use_encoded_value' and unknown_value=-1 handles new categories gracefully in single-sample inference.
feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = feature_encoder.fit_transform(X)

# For the target variable, we also need to encode it.
# We can use pandas factorize or another OrdinalEncoder.
# Let's use OrdinalEncoder for consistency.
target_encoder = OrdinalEncoder()
y_encoded = target_encoder.fit_transform(y.to_frame()).ravel() # .ravel() converts to 1D array

X_encoded_df = pd.DataFrame(X_encoded, columns=X.columns)
y_encoded_series = pd.Series(y_encoded, name='Play_Encoded')

print("Encoded Features (X_encoded) head:")
display(X_encoded_df.head())
print("\nEncoded Target (y_encoded) head:")
display(y_encoded_series.head())

# Store the categories for future reference (e.g., for inverse transformation)
print("\nFeature categories:")
for i, col in enumerate(X.columns):
    print(f"{col}: {feature_encoder.categories_[i]}")

print("\nTarget categories:")
print(f"Play: {target_encoder.categories_[0]}")

Encoded Features (X_encoded) head:


,Outlook,Temperature,Humidity,Wind
0,2.0,1.0,0.0,1.0
1,2.0,1.0,0.0,0.0
2,0.0,1.0,0.0,1.0
3,1.0,2.0,0.0,1.0
4,1.0,0.0,1.0,1.0



Encoded Target (y_encoded) head:


,Play_Encoded
0,0.0
1,0.0
2,1.0
3,1.0
4,1.0



Feature categories:
Outlook: ['Overcast' 'Rain' 'Sunny']
Temperature: ['Cool' 'Hot' 'Mild']
Humidity: ['High' 'Normal']
Wind: ['Strong' 'Weak']

Target categories:
Play: ['No' 'Yes']


### Dataset Partitioning: Divide the dataset into training and testing subsets.

In [ ]:
# Divide the dataset into training and testing subsets (80:20 split)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

Training features shape: (40, 4)
Testing features shape: (10, 4)
Training target shape: (40,)
Testing target shape: (10,)


### Naive Bayes Model Training & Evaluation:

In [ ]:
# Initialize and train the Categorical Naive Bayes model
cnb = CategoricalNB()
cnb.fit(X_train, y_train)

# Predict the class labels for the test dataset
y_pred_cnb = cnb.predict(X_test)

# Calculate and display the overall Model Accuracy
accuracy_cnb = accuracy_score(y_test, y_pred_cnb)
print(f"Categorical Naive Bayes Model Accuracy: {accuracy_cnb:.4f}")

# Display the Confusion Matrix
print("\nConfusion Matrix for Categorical Naive Bayes:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_cnb),
                     index=[f'Actual {i}' for i in target_encoder.categories_[0]],
                     columns=[f'Predicted {i}' for i in target_encoder.categories_[0]]))

# Display the Classification Report
print("\nClassification Report for Categorical Naive Bayes:")
print(classification_report(y_test, y_pred_cnb, target_names=target_encoder.categories_[0]))

Categorical Naive Bayes Model Accuracy: 0.8000

Confusion Matrix for Categorical Naive Bayes:


,Predicted No,Predicted Yes
Actual No,1,1
Actual Yes,1,7



Classification Report for Categorical Naive Bayes:
              precision    recall  f1-score   support

          No       0.50      0.50      0.50         2
         Yes       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



### Single-Sample Inference:

In [ ]:
# Define the single sample for prediction
single_sample = pd.DataFrame([
    {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
])

print("Single sample for prediction:")
display(single_sample)

# Encode the single sample using the previously fitted feature_encoder
# We need to handle potential unknown categories, though in this case, they should all be known.
encoded_single_sample = feature_encoder.transform(single_sample)

# Predict the class label
predicted_label_encoded = cnb.predict(encoded_single_sample)

# Predict the class probabilities
predicted_proba = cnb.predict_proba(encoded_single_sample)

# Inverse transform the predicted label to get the original categorical label
predicted_label_original = target_encoder.inverse_transform(predicted_label_encoded.reshape(-1, 1))

print(f"\nPredicted class label for the single sample: {predicted_label_original[0][0]}")
print(f"Corresponding class probabilities: {predicted_proba[0]}")

# Display probabilities with original class names
proba_df = pd.DataFrame(predicted_proba, columns=target_encoder.categories_[0])
print("\nClass Probabilities (with original labels):")
display(proba_df)

Single sample for prediction:


,Outlook,Temperature,Humidity,Wind
0,Sunny,Cool,High,Strong



Predicted class label for the single sample: No
Corresponding class probabilities: [0.92560203 0.07439797]

Class Probabilities (with original labels):


,No,Yes
0,0.925602,0.074398


### Model Comparison: Train Decision Tree, Logistic Regression, and SVM

In [ ]:
# Initialize and train models

# Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)
y_pred_dt = dt_classifier.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)

# Logistic Regression Classifier
# For categorical features encoded as ordinals, Logistic Regression can still work
# with proper scaling, but here we proceed without for simplicity as per common practice in similar scenarios.
# 'liblinear' solver works well for small datasets and handles L1/L2 penalties.
lr_classifier = LogisticRegression(random_state=42, solver='liblinear')
lr_classifier.fit(X_train, y_train)
y_pred_lr = lr_classifier.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

# Support Vector Machine (SVM) Classifier
# Using a linear kernel for simplicity. C is the regularization parameter.
svm_classifier = SVC(random_state=42, kernel='linear', probability=True) # probability=True to get predict_proba
svm_classifier.fit(X_train, y_train)
y_pred_svm = svm_classifier.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)

print(f"Decision Tree Classifier Accuracy: {accuracy_dt:.4f}")
print(f"Logistic Regression Classifier Accuracy: {accuracy_lr:.4f}")
print(f"SVM Classifier Accuracy: {accuracy_svm:.4f}")

Decision Tree Classifier Accuracy: 0.8000
Logistic Regression Classifier Accuracy: 0.4000
SVM Classifier Accuracy: 0.6000


#### Single-Sample Inference and Comparison for all models

In [ ]:
# Using the encoded_single_sample from previous step

results = []

# Naive Bayes (already calculated, just add to table)
predicted_label_cnb_orig = target_encoder.inverse_transform(cnb.predict(encoded_single_sample).reshape(-1, 1))[0][0]
predicted_proba_cnb = cnb.predict_proba(encoded_single_sample)[0]
results.append({
    'Model': 'Categorical Naive Bayes',
    'Test Accuracy': accuracy_cnb,
    'Predicted Label': predicted_label_cnb_orig,
    'Class Probabilities': dict(zip(target_encoder.categories_[0], predicted_proba_cnb))
})

# Decision Tree
predicted_label_dt_encoded = dt_classifier.predict(encoded_single_sample)
predicted_label_dt_orig = target_encoder.inverse_transform(predicted_label_dt_encoded.reshape(-1, 1))[0][0]
predicted_proba_dt = dt_classifier.predict_proba(encoded_single_sample)[0]
results.append({
    'Model': 'Decision Tree',
    'Test Accuracy': accuracy_dt,
    'Predicted Label': predicted_label_dt_orig,
    'Class Probabilities': dict(zip(target_encoder.categories_[0], predicted_proba_dt))
})

# Logistic Regression
predicted_label_lr_encoded = lr_classifier.predict(encoded_single_sample)
predicted_label_lr_orig = target_encoder.inverse_transform(predicted_label_lr_encoded.reshape(-1, 1))[0][0]
predicted_proba_lr = lr_classifier.predict_proba(encoded_single_sample)[0]
results.append({
    'Model': 'Logistic Regression',
    'Test Accuracy': accuracy_lr,
    'Predicted Label': predicted_label_lr_orig,
    'Class Probabilities': dict(zip(target_encoder.categories_[0], predicted_proba_lr))
})

# SVM
predicted_label_svm_encoded = svm_classifier.predict(encoded_single_sample)
predicted_label_svm_orig = target_encoder.inverse_transform(predicted_label_svm_encoded.reshape(-1, 1))[0][0]
predicted_proba_svm = svm_classifier.predict_proba(encoded_single_sample)[0]
results.append({
    'Model': 'SVM',
    'Test Accuracy': accuracy_svm,
    'Predicted Label': predicted_label_svm_orig,
    'Class Probabilities': dict(zip(target_encoder.categories_[0], predicted_proba_svm))
})

comparison_df = pd.DataFrame(results)
print("\nModel Comparison Table:")
display(comparison_df.set_index('Model'))


Model Comparison Table:


,Test Accuracy,Predicted Label,Class Probabilities
Model,,,
Categorical Naive Bayes,0.8,No,"{'No': 0.9256020301956863, 'Yes': 0.0743979698..."
Decision Tree,0.8,No,"{'No': 1.0, 'Yes': 0.0}"
Logistic Regression,0.4,No,"{'No': 0.9466476048933914, 'Yes': 0.0533523951..."
SVM,0.6,No,"{'No': 0.9622230337005967, 'Yes': 0.0377769662..."


### Analysis Report

The models produced different predictions and probability scores for the given test instance due to their inherent algorithmic differences and assumptions about the data. Naive Bayes, assuming independence of features, might perform well if this assumption holds. Decision Trees make decisions based on feature splits, potentially overfitting or underfitting depending on complexity. Logistic Regression models the probability using a linear combination of features, while SVM aims to find an optimal hyperplane separating classes. These varied approaches lead to different interpretations of the input features and thus, different outputs, especially for a single, potentially ambiguous, sample.